In [1]:
import coiled
import fsspec
import s3fs
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging 
import numpy as np
import pytz
import dask
import re
import requests
import sparse
import time
import warnings
import zarr
from io import BytesIO
from datetime import datetime
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy
from flox.xarray import xarray_reduce
import pygwalker as pyg

# T0 INSTALL FLOX:
# Downloaded flox 0.10.3 from https://pypi.org/project/flox/#files usin the source distribution (tar.gz, https://files.pythonhosted.org/packages/0b/b6/5e3d79ef8e3dd3bb1ba656167e2059c0aa000244321995c508db32c7a578/flox-0.10.3.tar.gz)
# Installed in my working conda environment using pip: pip install /home/dagibbs22/flox-0.10.3.tar.gz

# TO CREATE A NOTEBOOK IN A COILED CLUSTER
# coiled notebook start --region=us-east-1

In [ ]:
# Zarr creation cluster
cluster = coiled.Cluster(
    name="LULUCF_zarr_creation",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=5,
    tags={"project": "LULUCF_zonal_stats"},
    scheduler_vm_types="r7g.xlarge", 
    worker_vm_types="r7g.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

In [71]:
# Zonal stats cluster
cluster = coiled.Cluster(
    name="LULUCF_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=20,
    tags={"project": "LULUCF_zonal_stats"},
    scheduler_vm_types="r7g.xlarge", 
    worker_vm_types="r7g.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                ╷                                                             │
│   Package      │ Note                                                        │
│ ╶──────────────┼───────────────────────────────────────────────────────────╴ │
│   flox         │ Wheel built from ~/flox-0.10.3.tar.gz                       │
│                ╵                                                             │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│                 ╷                                                ╷           │
│   Package       │ Error                                          │ Level     │
│ ╶───────────────┼────────────────────────────────────────────────┼─────────╴ │
│   pygwalker     │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ duckdb==0.9.2, but you have duckdb 1.3.0.      │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ segment-analytics-python==2.2.3, but you have  │           │
│                 │ segment-analytics-python 2.3.3.                │           │
│   pydantic_core │ pydantic-core~=2.33.2 has no install candidate │ Warning   │
│                 │ for Python 3.12 linux-aarch64 on conda-forge   │           │
│   dtale         │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ dtale 3.17.0 has requirement dash<=2.18.2;     │           │
│                 │ python_version > "3.7", but you have dash      │           │
│                 │ 3.0.4.                                         │           │
│                 │ dtale 3.17.0 has requirement                   │           │
│                 │ dash-bootstrap-components<=1.7.1;              │           │
│                 │ python_version > "3.0", but you have           │           │
│                 │ dash-bootstrap-components 2.0.3.               │           │
│                 │ dtale 3.17.0 has requirement dash_daq<=0.5.0,  │           │
│                 │ but you have dash-daq 0.6.0.                   │           │
│   awscrt        │ awscrt~=0.26.1 has no install candidate for    │ Warning   │
│                 │ Python 3.12 linux-aarch64 on conda-forge       │           │
│                 ╵                                                ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

In [ ]:
client.restart() 

In [ ]:
local_cluster = LocalCluster()  
local_client = Client(local_cluster)
local_client

In [ ]:
local_client.shutdown()

In [3]:
# To hide the warning "UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future."
warnings.filterwarnings(
    "ignore",
    message="Consolidated metadata is currently not part in the Zarr format 3 specification.*",
    category=UserWarning,
    module="zarr.api.asynchronous"
)

# Conversion of carbon to CO2
C_to_CO2 = 44/12

def timestr():

    # Define the Eastern Time timezone
    eastern = pytz.timezone('US/Eastern')

    # Get the current time in UTC and convert to Eastern Time
    eastern_time = datetime.now(eastern)

    # Format the time as a string
    return eastern_time.strftime("%Y%m%d_%H_%M_%S")

In [4]:
# Creates a Pandas dataframe with the state_nodes codes and meanings from an Excel spreadsheet
def create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet_name):

    try:
        # Tries fetching the file from the S3 URL
        # print(f"Attempting to download file from URL: {spreadsheet}")
        response = requests.get(state_node_lookup_table_s3, timeout=10)
        response.raise_for_status()
        state_node_df = pd.read_excel(BytesIO(response.content), sheet_name=sheet_name)

    except (requests.exceptions.RequestException, Exception) as e:
        print(f"Failed to download file from S3. Falling back to local file. Error: {e}")

        print(f"Reading file from local path: {state_node_lookup_table_local}")
        state_node_df = pd.read_excel(state_node_lookup_table_local, sheet_name=sheet_name)

    return state_node_df

In [59]:
# Lists uris in an s3 folder, for creating zarr of them
# per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682201ec-1f84-800a-a9f9-c9564f613208
def list_folder_uris(base_uri):

    # Initializes S3 filesystem
    fs = s3fs.S3FileSystem(anon=False)  # Set anon=True if public bucket
    
    # Lists all files in the directory
    all_files = fs.ls(base_uri)
    
    # Filters for GeoTIFFs
    tif_files = [f"s3://{f}" for f in all_files if f.endswith(".tif")]
    
    # Converts to a Pandas Series
    series = pd.Series(tif_files)
    
    return series

# Extracts file pattern from uri. Assumes that file pattern includes _ha_yr (as it does from the LULUCF model).
def parse_pattern_from_uri(uri_series):

    uri = uri_series.values.tolist()[0]
    # print("Parsing URI:", uri)

    # regex per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/681a538d-55e4-800a-818b-bcf850757ba0
    pattern = r"__([a-zA-Z0-9_]+(?:__?[a-zA-Z0-9_]+)*)_ha_yr_\d{4}_\d{4}\.tif$"
    match = re.search(pattern, uri)

    if match:
        return match.group(1)
    else:
        return None

In [6]:
# Makes xarray dataframe (I think not a dataset) from list of s3 uris.
# This came from Solomon Negusse and I haven't really changed it.
# He said that an online forum suggested using xr.openmfdataset to open non-overlapping geotifs.
def make_xarray_chunks(tile_uris, chunk_size):

    xarray_chunks = xr.open_mfdataset(
        tile_uris.values.tolist(),
        parallel=True,
        chunks={'x': chunk_size, 'y':chunk_size}
    ).squeeze()
    # ).squeeze().persists()  # Need this if reading from geotifs directly, rather then creating zarrs

    return xarray_chunks

In [7]:
# Crops one input to the other input's extent.
# ref is the reference dataset that is being cropped to. 
# From long chat in https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/684749fe-7b30-800a-ba8b-c502377f2c3a
def safe_crop(ds, ref):
    return ds.sel(x=ref.x, y=ref.y, method="nearest")

In [8]:
### Options for contextual layer values.
### Every contextual layer needs to have all possible values listed here.

# GADM v4.1 adm0 IDs (from Solomon Negusse's notebook)
gadm_adm0_ids = np.array([  0.,   4.,   8.,  10.,  12.,  16.,  20.,  24.,  28.,  31.,  32.,
        36.,  40.,  44.,  48.,  50.,  51.,  52.,  56.,  60.,  64.,  68.,
        70.,  72.,  74.,  76.,  84.,  86.,  90.,  92.,  96., 100., 104.,
       108., 112., 116., 120., 124., 132., 136., 140., 144., 148., 152.,
       156., 158., 162., 166., 170., 174., 175., 178., 180., 184., 188.,
       191., 192., 196., 203., 204., 208., 212., 214., 218., 222., 226.,
       231., 232., 233., 234., 238., 239., 242., 246., 248., 250., 254.,
       258., 260., 262., 266., 268., 270., 275., 276., 288., 292., 296.,
       300., 304., 308., 312., 316., 320., 324., 328., 332., 334., 336.,
       340., 348., 352., 356., 360., 364., 368., 372., 376., 380., 384.,
       388., 392., 398., 400., 404., 408., 410., 414., 417., 418., 422.,
       426., 428., 430., 434., 438., 440., 442., 450., 454., 458., 462.,
       466., 470., 474., 478., 480., 484., 492., 496., 498., 499., 500.,
       504., 508., 512., 516., 520., 524., 528., 531., 533., 534., 535.,
       540., 548., 554., 558., 562., 566.,70., 574., 578., 580., 581.,
       583., 584., 585., 586., 591., 598., 600., 604., 608., 612., 616.,
       620., 624., 626., 630., 634., 638., 642., 643., 646., 652., 654.,
       659., 660., 662., 663., 666., 670., 674., 678., 682., 686., 688.,
       690., 694., 702., 703., 704., 705., 706., 710., 716., 724., 728.,
       729., 732., 740., 744., 748., 752., 756., 760., 762., 764., 768.,
       772., 776., 780., 784., 788., 792., 795., 796., 798., 800., 804.,
       807., 818., 826., 831., 832., 833., 834., 840., 850., 854., 858.,
       860., 862., 876., 882., 887., 894.], dtype=np.uint16)

# Primary forest value options
primary_forest_IFL_codes = np.array([0, 1], dtype=np.uint8)

# From https://github.com/wri/project-zeno-data-infra/blob/main/notebooks/grasslands_areas_gadm_2000-2022.ipynb
numeric_to_alpha3 = {
    4: 'AFG', 248: 'ALA', 8: 'ALB', 12: 'DZA', 16: 'ASM', 20: 'AND', 24: 'AGO', 660: 'AIA',
    10: 'ATA', 28: 'ATG', 32: 'ARG', 51: 'ARM', 533: 'ABW', 36: 'AUS', 40: 'AUT', 31: 'AZE',
    44: 'BHS', 48: 'BHR', 50: 'BGD', 52: 'BRB', 112: 'BLR', 56: 'BEL', 84: 'BLZ', 204: 'BEN',
    60: 'BMU', 64: 'BTN', 68: 'BOL', 535: 'BES', 70: 'BIH', 72: 'BWA', 74: 'BVT', 76: 'BRA',
    86: 'IOT', 96: 'BRN', 100: 'BGR', 854: 'BFA', 108: 'BDI', 132: 'CPV', 116: 'KHM', 120: 'CMR',
    124: 'CAN', 136: 'CYM', 140: 'CAF', 148: 'TCD', 152: 'CHL', 156: 'CHN', 162: 'CXR', 166: 'CCK',
    170: 'COL', 174: 'COM', 178: 'COG', 180: 'COD', 184: 'COK', 188: 'CRI', 384: 'CIV', 191: 'HRV',
    192: 'CUB', 531: 'CUW', 196: 'CYP', 203: 'CZE', 208: 'DNK', 262: 'DJI', 212: 'DMA', 214: 'DOM',
    218: 'ECU', 818: 'EGY', 222: 'SLV', 226: 'GNQ', 232: 'ERI', 233: 'EST', 748: 'SWZ', 231: 'ETH',
    238: 'FLK', 234: 'FRO', 242: 'FJI', 246: 'FIN', 250: 'FRA', 254: 'GUF', 258: 'PYF', 260: 'ATF',
    266: 'GAB', 270: 'GMB', 268: 'GEO', 276: 'DEU', 288: 'GHA', 292: 'GIB', 300: 'GRC', 304: 'GRL',
    308: 'GRD', 312: 'GLP', 316: 'GUM', 320: 'GTM', 831: 'GGY', 324: 'GIN', 624: 'GNB', 328: 'GUY',
    332: 'HTI', 334: 'HMD', 336: 'VAT', 340: 'HND', 344: 'HKG', 348: 'HUN', 352: 'ISL', 356: 'IND',
    360: 'IDN', 364: 'IRN', 368: 'IRQ', 372: 'IRL', 833: 'IMN', 376: 'ISR', 380: 'ITA', 388: 'JAM',
    392: 'JPN', 832: 'JEY', 400: 'JOR', 398: 'KAZ', 404: 'KEN', 296: 'KIR', 408: 'PRK', 410: 'KOR',
    414: 'KWT', 417: 'KGZ', 418: 'LAO', 428: 'LVA', 422: 'LBN', 426: 'LSO', 430: 'LBR', 434: 'LBY',
    438: 'LIE', 440: 'LTU', 442: 'LUX', 446: 'MAC', 450: 'MDG', 454: 'MWI', 458: 'MYS', 462: 'MDV',
    466: 'MLI', 470: 'MLT', 584: 'MHL', 474: 'MTQ', 478: 'MRT', 480: 'MUS', 175: 'MYT', 484: 'MEX',
    583: 'FSM', 498: 'MDA', 492: 'MCO', 496: 'MNG', 499: 'MNE', 500: 'MSR', 504: 'MAR', 508: 'MOZ',
    104: 'MMR', 516: 'NAM', 520: 'NRU', 524: 'NPL', 528: 'NLD', 540: 'NCL', 554: 'NZL', 558: 'NIC',
    562: 'NER', 566: 'NGA', 570: 'NIU', 574: 'NFK', 807: 'MKD', 580: 'MNP', 578: 'NOR', 512: 'OMN',
    586: 'PAK', 585: 'PLW', 275: 'PSE', 591: 'PAN', 598: 'PNG', 600: 'PRY', 604: 'PER', 608: 'PHL',
    612: 'PCN', 616: 'POL', 620: 'PRT', 630: 'PRI', 634: 'QAT', 638: 'REU', 642: 'ROU', 643: 'RUS',
    646: 'RWA', 652: 'BLM', 654: 'SHN', 659: 'KNA', 662: 'LCA', 663: 'MAF', 666: 'SPM', 670: 'VCT',
    882: 'WSM', 674: 'SMR', 678: 'STP', 682: 'SAU', 686: 'SEN', 688: 'SRB', 690: 'SYC', 694: 'SLE',
    702: 'SGP', 534: 'SXM', 703: 'SVK', 705: 'SVN', 90: 'SLB', 706: 'SOM', 710: 'ZAF', 239: 'SGS',
    728: 'SSD', 724: 'ESP', 144: 'LKA', 729: 'SDN', 740: 'SUR', 744: 'SJM', 752: 'SWE', 756: 'CHE',
    760: 'SYR', 158: 'TWN', 762: 'TJK', 834: 'TZA', 764: 'THA', 626: 'TLS', 768: 'TGO', 772: 'TKL',
    776: 'TON', 780: 'TTO', 788: 'TUN', 792: 'TUR', 795: 'TKM', 796: 'TCA', 798: 'TUV', 800: 'UGA',
    804: 'UKR', 784: 'ARE', 826: 'GBR', 840: 'USA', 581: 'UMI', 858: 'URY', 860: 'UZB', 548: 'VUT',
    862: 'VEN', 704: 'VNM', 92: 'VGB', 850: 'VIR', 876: 'WLF', 732: 'ESH', 887: 'YEM', 894: 'ZMB',
    716: 'ZWE'
}

In [9]:
# Converts results of flox to coordinate dictionary.
# This code came from Solomon Negusse and I haven't changed it in any substantial way.
def convert_to_coord_dict(flux_results):

    print(f"   Postprocessing {interval}: {timestr()}")
    sparse_data = flux_results.data
    
    dim_names = flux_results.dims
    indices = sparse_data.coords  # tuple of arrays with indices into each dim
    values = sparse_data.data     # non-zero values
    
    coord_dict = {
        dim: flux_results.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    coord_dict["value"] = values

    return coord_dict

In [10]:
# Converts flox output to dataframe and does some processing of it:
# replaces the numeric flux type with the name
# classifies specific flux types to larger groupings
# adds the interval end year to the dataframe
# adds the state node meaning to the dataframe
# converts area from m^2 to ha
def create_interval_df(coord_dict, state_node_df):

    df = pd.DataFrame(coord_dict)
    # print(df)

    # Replaces numeric values for output flux types with their names for ease of interpretation
    df['analysis_layer'] = df['analysis_layer'].replace(analysis_layer_dict)
    # print("with analysis_layer:", df)

    # Adds the interval end year to the dataframe
    df['interval_end'] = interval_end_year
    # print("with interval end year:", df)

    # Adds the state_node meaning and classifications to the dataframe
    df = df.merge(state_node_df[['state_nodes', 'meaning', 'broad_class', 'detailed_class']],
              left_on='state_nodes', right_on='state_nodes',
              how='left')
    # print("merged:", df)

    # # Converts area from m^2 to ha
    # df.loc[df['analysis_layer'].eq('area__ha'), 'value'] = df['value'] / 10000   
    # print("final:", df)

    # # From https://github.com/wri/project-zeno-data-infra/blob/main/notebooks/grasslands_areas_gadm_2000-2022.ipynb
    # df['countries'] = df.countries.apply(lambda x: numeric_to_alpha3[x])

    return df

In [11]:
# Calculates flux densities (Mg CO2 or CO2e/ha) for each output flux
# Per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682a8c76-0618-800a-a201-18fd404a281f
def calculate_interval_flux_densities(df):

    # Step 1: Filters out area and flux data
    area_df = df[df['analysis_layer'] == 'area__ha'].copy()
    flux_df = df[df['analysis_layer'] != 'area__ha'].copy()
    
    # Step 2: Merges flux data with area data on matching keys (contextual_layer_names)
    merged = pd.merge(
        flux_df,
        area_df[contextual_layer_names + ['interval_end', 'value']],
        on=contextual_layer_names,
        how='left',
        suffixes=('', '_area')
    )
    # print("merged:", merged)
    
    # Step 3: Computes per-hectare flux (converts CO2 to C)
    merged['value_per_ha'] = merged['value'] / merged['value_area'] / C_to_CO2
    
    # Step 4: Prepares flux density rows to append
    new_rows = merged.copy()
    new_rows['analysis_layer'] = new_rows['analysis_layer'] + '__C_per_ha'
    new_rows['value'] = new_rows['value_per_ha']
    new_rows = new_rows.drop(columns=['value_area', 'interval_end_area', 'value_per_ha'])
    # print("new rows:", new_rows)
    
    # Step 5: Appends flux density rows to original dataframe
    result_df = pd.concat([df, new_rows], ignore_index=True)

    return result_df

Code to run zonal stats

In [51]:
# General zonal stat run properties

model_version = "version_0_4_2"  # model version, from s3 paths that are being read
run_date = "20250806"   # model run date, from s3 paths that are being read
chunk_size = 10000  # pixels

# s3 folders for model outputs being analyzed
output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"  # Model output path, for inputs to zonal stats

# Analysis layer s3 paths
gross_emis_CO2_folder = f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/hybrid_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_emis_all_gases_folder = f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/hybrid_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_remv_all_pools_folder = f"{output_path}gross_removals__all_C_pools__MgCO2/standard_model/hybrid_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
net_flux_all_pools_CO2_folder = f"{output_path}net_flux__all_C_pools__CO2_only__MgCO2/standard_model/hybrid_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
net_flux_all_pools_all_gases_folder = f"{output_path}net_flux__all_C_pools__all_gases__MgCO2e/standard_model/hybrid_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
node_folder = f"{output_path}land_state_node/standard_model/hybrid_intervals/INTERVAL/4000_pixels/{run_date}/"

# Folder where the model output zarrs are stored. They are in their own special outputs folder (at least for now-- we could change this)
zarr_s3_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}_1x1_inputs_per_ha/zarr/{run_date}/"

# zarrs for layers not from the flux model (only need to created once)
# They are in a central folder, not with their specific geotif tile sets (at least for now-- we could change this)
adm0_folder = "s3://gfw2-data/gadm_administrative_boundaries/v4.1/v4.1.64__from_gfw-data-lake/raster/epsg-4326/10/40000/adm0/gdal-geotiff/" #GADM v4.1
adm0_zarr_name = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/contextual_layer_global_zarr/GADM4_1_adm0_global/20250604/global_GADM41_adm0_20250604.zarr"

pixel_area_folder = "s3://gfw2-data/analyses/umd_area_2013__from_gfw-data-lake/v1.10/raster/epsg-4326/10/40000/area_m/gdal-geotiff/"
pixel_area_zarr_name = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/contextual_layer_global_zarr/pixel_area/20250730/global_pixel_area_20250730.zarr"

primary_forest_IFL_folder = "s3://gfw2-data/climate/carbon_model/ifl_primary_merged/processed/20200724/"
primary_forest_IFL_zarr_name = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/contextual_layer_global_zarr/IFL2000_tropical_primary_forest_2001/20250806/ifl_primary_forest_merged.zarr"


# Spreadsheet for state_node meanings (local computer and s3 locations)
state_node_lookup_table_local = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/LULUCF_state_node_lookup_table.xlsx"
state_node_lookup_table_s3 = "http://gfw2-data.s3.amazonaws.com/climate/AFOLU_flux_model/LULUCF/state_node_lookup_tables/LULUCF_state_node_lookup_table.xlsx"
sheet = "v042_20250805"

In [ ]:
# %%time

## CREATES ZARRS FOR INPUTS NOT GENERATED BY THE AFOLU MODEL
## THIS SHOULD ONLY EVER HAVE TO BE DONE ONCE FOR EACH INPUT

# print(f"Reading inputs that apply to all intervals: {timestr()}")

# # GADM adm0
# adm0_uris = list_folder_uris(adm0_folder)
# print("adm0_folder:", adm0_folder)
# print(adm0_uris[0])
# print(f"Tile count in {adm0_folder}: {len(adm0_uris)}")

# print(f"   Reading adm0: {timestr()}")
# adm0_xarray_chunks = make_xarray_chunks(adm0_uris, chunk_size)
# adm0_xarray_chunks['band_data'] = adm0_xarray_chunks['band_data'].astype('uint16')  # adm0 should be uint16 but make_xarray_chunks makes it float64 for some reason
# # print("adm0_xarray_chunks:", adm0_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# # print(adm0_xarray_chunks)
# print("X resolution:", np.diff(adm0_xarray_chunks.x.values).mean())
# print("Y resolution:", np.diff(adm0_xarray_chunks.y.values).mean())
# print("X range:", adm0_xarray_chunks.x.values[[0, -1]])
# print("Y range:", adm0_xarray_chunks.y.values[[0, -1]])
# print("Shape (y, x):", adm0_xarray_chunks.sizes['y'], adm0_xarray_chunks.sizes['x'])

# print(f"   zarring adm0: {timestr()}")
# adm0_xarray_chunks.to_zarr(adm0_zarr_name, mode='w')
# print(f"   Finished zarring adm0: {timestr()}")


# # Pixel area
# pixel_area_uris = list_folder_uris(pixel_area_folder)
# print("pixel_area_folder:", pixel_area_folder)
# print(pixel_area_uris[0])
# print(f"Tile count in {pixel_area_folder}: {len(pixel_area_uris)}")

# print(f"   Reading pixel_area: {timestr()}")
# pixel_area_xarray_chunks = make_xarray_chunks(pixel_area_uris, chunk_size)
# print("pixel_area_xarray_chunks:", pixel_area_xarray_chunks)

# print(f"   zarring pixel_area: {timestr()}")
# pixel_area_xarray_chunks.to_zarr(pixel_area_zarr_name, mode='w')
# print(f"   Finished zarring pixel_area: {timestr()}")


# # Humid tropical primary forest/IFL merged
# primary_forest_IFL_uris = list_folder_uris(primary_forest_IFL_folder)
# print("primary_forest_IFL_folder:", primary_forest_IFL_folder)
# print(primary_forest_IFL_uris[0])
# print(f"Tile count in {primary_forest_IFL_folder}: {len(primary_forest_IFL_uris)}")

# print(f"   Reading primary_forest_IFL: {timestr()}")
# primary_forest_IFL_xarray_chunks = make_xarray_chunks(primary_forest_IFL_uris, chunk_size)
# primary_forest_IFL_xarray_chunks['band_data'] = primary_forest_IFL_xarray_chunks['band_data'].astype('uint8')  # should be uint8 but make_xarray_chunks makes it float32 for some reason
# print("primary_forest_IFL_xarray_chunks:", primary_forest_IFL_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring primary_forest_IFL: {timestr()}")
# primary_forest_IFL_xarray_chunks.to_zarr(primary_forest_IFL_zarr_name, mode='w')
# print(f"   Finished zarring primary_forest_IFL: {timestr()}")

In [14]:
%%time

### CONVERTS GEOTIFS TO ZARRS AND STORES THEM IN S3
### ONLY NEED TO DO THIS THE FIRST TIME RUNNING AN ANALYSIS ON MODEL OUTPUTS

# interval_end_years = [2005]
interval_end_years = [2005, 2010, 2015]
# interval_end_years = [2024]
# interval_end_years = [2016, 2017, 2018]
# interval_end_years = [2019, 2020, 2021, 2022, 2023]
# interval_end_years = [2005, 2010, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

analysis_start_time = time.time()

for interval_end_year in interval_end_years:

    # Sets the interval name based on the interval type (5-year or annual)
    if interval_end_year in [2005, 2010, 2015]:
        interval = f"{interval_end_year-4}_{interval_end_year}"
    else:
        interval = f"{interval_end_year-1}_{interval_end_year}"

    print(f"Processing {interval}: {timestr()}")
    interval_start_time = time.time()

    # Creates a Pandas series of s3 uris for this specific analysis layer
    gross_emis_CO2_folder_interval = gross_emis_CO2_folder.replace("INTERVAL", interval)
    gross_emis_CO2_uris = list_folder_uris(gross_emis_CO2_folder_interval)

    gross_emis_all_gases_folder_interval = gross_emis_all_gases_folder.replace("INTERVAL", interval)
    gross_emis_all_gases_uris = list_folder_uris(gross_emis_all_gases_folder_interval)
    
    gross_remv_all_pools_folder_interval = gross_remv_all_pools_folder.replace("INTERVAL", interval)
    gross_remv_all_pools_uris = list_folder_uris(gross_remv_all_pools_folder_interval)
    
    net_flux_all_pools_CO2_folder_interval = net_flux_all_pools_CO2_folder.replace("INTERVAL", interval)
    net_flux_all_pools_CO2_uris = list_folder_uris(net_flux_all_pools_CO2_folder_interval)

    net_flux_all_pools_all_gases_folder_interval = net_flux_all_pools_all_gases_folder.replace("INTERVAL", interval)
    net_flux_all_pools_all_gases_uris = list_folder_uris(net_flux_all_pools_all_gases_folder_interval)
    
    node_folder_interval = node_folder.replace("INTERVAL", interval)
    node_tile_year_uris = list_folder_uris(node_folder_interval)

    
    print("    gross_emis_CO2_folder_interval:", gross_emis_CO2_folder_interval)
    print(gross_emis_CO2_uris[0])
    print(f"    Tile count in {gross_emis_CO2_folder_interval}: {len(gross_emis_CO2_uris)}")
    
    print("    gross_emis_all_gases_folder_interval:", gross_emis_all_gases_folder_interval)
    print(gross_emis_all_gases_uris[0])
    print(f"    Tile count in {gross_emis_all_gases_folder_interval}: {len(gross_emis_all_gases_uris)}")
    
    print("    gross_remv_all_pools_folder_interval:", gross_remv_all_pools_folder_interval)
    print(gross_remv_all_pools_uris[0])
    print(f"    Tile count in {gross_remv_all_pools_folder_interval}: {len(gross_remv_all_pools_uris)}")

    print("    net_flux_all_pools_CO2_folder_interval:", net_flux_all_pools_CO2_folder_interval)
    print(net_flux_all_pools_CO2_uris[0])
    print(f"    Tile count in {net_flux_all_pools_CO2_folder_interval}: {len(net_flux_all_pools_CO2_uris)}")

    print("    net_flux_all_pools_all_gases_folder_interval:", net_flux_all_pools_all_gases_folder_interval)
    print(net_flux_all_pools_all_gases_uris[0])
    print(f"    Tile count in {net_flux_all_pools_all_gases_folder_interval}: {len(net_flux_all_pools_all_gases_uris)}")

    print("    node_folder_interval:", node_folder_interval)
    print(node_tile_year_uris[0])
    print(f"    Tile count in {node_folder_interval}: {len(node_tile_year_uris)}")

    
    print(f"   Reading gross emis CO2 only for {interval}: {timestr()}")
    gross_emis_CO2_xarray_chunks = make_xarray_chunks(gross_emis_CO2_uris, chunk_size)
    print(f"   Reading gross emis all gases for {interval}: {timestr()}")
    gross_emis_all_gases_xarray_chunks = make_xarray_chunks(gross_emis_all_gases_uris, chunk_size)   
    print(f"   Reading gross removals for {interval}: {timestr()}")
    gross_remv_all_pools_xarray_chunks = make_xarray_chunks(gross_remv_all_pools_uris, chunk_size)    
    print(f"   Reading net flux CO2 only for {interval}: {timestr()}")
    net_flux_all_pools_CO2_xarray_chunks = make_xarray_chunks(net_flux_all_pools_CO2_uris, chunk_size)
    print(f"   Reading net flux all gases for {interval}: {timestr()}")
    net_flux_all_pools_all_gases_xarray_chunks = make_xarray_chunks(net_flux_all_pools_all_gases_uris, chunk_size)   
    print(f"   Reading state_nodes for {interval}: {timestr()}")
    node_xarray_chunks = make_xarray_chunks(node_tile_year_uris, chunk_size)
    node_xarray_chunks['band_data'] = node_xarray_chunks['band_data'].astype('uint32')  # state_nodes should be uint32 but make_xarray_chunks makes it float64 for some reason

    # Names the analysis layers so it's easier to understand their metadata when printed below
    gross_emis_CO2_xarray_chunks.attrs['name'] = "gross_emis_CO2"
    gross_emis_all_gases_xarray_chunks.attrs['name'] = "gross_emis_all_gases"
    gross_remv_all_pools_xarray_chunks.attrs['name'] = "gross_remv_all_pools"
    net_flux_all_pools_CO2_xarray_chunks.attrs['name'] = "net_flux_all_pools"
    net_flux_all_pools_all_gases_xarray_chunks.attrs['name'] = "net_flux_all_pools_all_gases"
    node_xarray_chunks.attrs['name'] = "state_nodes"
  
    
    # per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/68309b36-0f48-800a-bd56-67180b55106e
    gross_emis_CO2_zarr_name = f"{zarr_s3_path}{interval}/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_{interval}.zarr"
    gross_emis_all_gases_zarr_name = f"{zarr_s3_path}{interval}/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_{interval}.zarr"
    gross_remv_all_pools_zarr_name = f"{zarr_s3_path}{interval}/gross_removals__all_C_pools__MgCO2_ha_yr_{interval}.zarr"
    net_flux_all_pools_CO2_zarr_name = f"{zarr_s3_path}{interval}/net_flux__all_C_pools__CO2_only__MgCO2_ha_yr_{interval}.zarr"
    net_flux_all_pools_all_gases_zarr_name = f"{zarr_s3_path}{interval}/net_flux__all_C_pools__all_gases__MgCO2e_ha_yr_{interval}.zarr"
    node_zarr_name = f"{zarr_s3_path}{interval}/land_state_node_{interval}.zarr"

    # Re-chunks zarrs from 4000x4000 pixels (geotif dimensions) to 10000x10000 pixels
    # per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6899f498-9e9c-8325-aecb-d1d8df7fe48f
    # Looked into how to avoid the large task graph warning but ChatGPT solutions required using zarr v2 instead of zarr v3. 
    target_chunks = {'x': chunk_size, 'y': chunk_size}
    gross_emis_CO2_xarray_chunks = gross_emis_CO2_xarray_chunks.chunk(target_chunks)
    gross_emis_all_gases_xarray_chunks = gross_emis_all_gases_xarray_chunks.chunk(target_chunks)
    gross_remv_all_pools_xarray_chunks = gross_remv_all_pools_xarray_chunks.chunk(target_chunks)
    net_flux_all_pools_CO2_xarray_chunks = net_flux_all_pools_CO2_xarray_chunks.chunk(target_chunks)
    net_flux_all_pools_all_gases_xarray_chunks = net_flux_all_pools_all_gases_xarray_chunks.chunk(target_chunks)
    node_xarray_chunks = node_xarray_chunks.chunk(target_chunks)

    analysis_layers = [
        gross_emis_CO2_xarray_chunks,
        gross_emis_all_gases_xarray_chunks,
        gross_remv_all_pools_xarray_chunks,
        net_flux_all_pools_CO2_xarray_chunks,
        net_flux_all_pools_all_gases_xarray_chunks,
        node_xarray_chunks
    ]

    for ds in analysis_layers:
        print(f"Metadata for {ds.attrs['name']}")
        print(f"    X, Y resolution: {np.diff(ds.x.values).mean(), np.diff(ds.y.values).mean()}")
        print(f"    X range: {ds.x.values[[0, -1]]}")
        print(f"    Y range: {ds.y.values[[0, -1]]}")
        print(f"    Shape (x, y): {ds.sizes['x'], ds.sizes['y']}")
        print(f"    Chunk size: {ds.chunks}")

    # Creates the zarrs and uploads them to s3. This is the part of the process that takes longest. 
    print(f"   zarring gross emis CO2 only for {interval}: {timestr()}")
    gross_emis_CO2_xarray_chunks.to_zarr(gross_emis_CO2_zarr_name, mode='w')
    print(f"   zarring gross emis all_gases only for {interval}: {timestr()}")
    gross_emis_all_gases_xarray_chunks.to_zarr(gross_emis_all_gases_zarr_name, mode='w')
    print(f"   zarring gross removals for {interval}: {timestr()}")
    gross_remv_all_pools_xarray_chunks.to_zarr(gross_remv_all_pools_zarr_name, mode='w')
    print(f"   zarring net flux CO2 only for {interval}: {timestr()}")
    net_flux_all_pools_CO2_xarray_chunks.to_zarr(net_flux_all_pools_CO2_zarr_name, mode='w')
    print(f"   zarring net flux all_gases for {interval}: {timestr()}")
    net_flux_all_pools_all_gases_xarray_chunks.to_zarr(net_flux_all_pools_all_gases_zarr_name, mode='w')
    print(f"   zarring state_nodes for {interval}: {timestr()}")
    node_xarray_chunks.to_zarr(node_zarr_name, mode='w')

    print(f"    Done zarring: {timestr()}")
    interval_end_time = time.time()
    print(f"   {interval} took {round(interval_end_time - interval_start_time)} seconds")

print(f"Done zarring all intervals: {timestr()}")
analysis_end_time = time.time()
print(f"Analysis took {round(analysis_end_time - analysis_start_time)} seconds")

client.shutdown()

Processing 2001_2005: 20250811_11_01_11
    gross_emis_CO2_folder_interval: s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_4_2/gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/hybrid_intervals/2001_2005/_ha_yr/4000_pixels/20250806/
s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_4_2/gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/hybrid_intervals/2001_2005/_ha_yr/4000_pixels/20250806/00N_050W__-44_-4_-43_-3__gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2001_2005.tif
    Tile count in s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_4_2/gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/hybrid_intervals/2001_2005/_ha_yr/4000_pixels/20250806/: 174
    gross_emis_all_gases_folder_interval: s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_4_2/gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/hybrid_intervals/2001_2005/_ha_yr/4000_pixels/20250806/
s3://gfw2-data/climate/

/home/dagibbs22/miniforge3/envs/coiled_20250606/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 10.27 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


   zarring gross emis all_gases only for 2001_2005: 20250811_11_01_53


/home/dagibbs22/miniforge3/envs/coiled_20250606/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 10.27 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


   zarring gross removals for 2001_2005: 20250811_11_02_04


...

   zarring net flux all_gases for 2011_2015: 20250811_11_05_40


/home/dagibbs22/miniforge3/envs/coiled_20250606/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 10.26 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


   zarring state_nodes for 2011_2015: 20250811_11_05_52


/home/dagibbs22/miniforge3/envs/coiled_20250606/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 10.25 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


    Done zarring: 20250811_11_06_03
   2011_2015 took 93 seconds
Done zarring all intervals: 20250811_11_06_03
Analysis took 292 seconds
CPU times: user 35.9 s, sys: 1.35 s, total: 37.3 s
Wall time: 4min 53s


In [72]:
%%time

### RUNS ZONAL STATISTICS ANALYSIS

combined_df = pd.DataFrame()  # dataframe for outputs across all model intervals
analysis_start_time = time.time()

# interval_end_years = [2005]
interval_end_years = [2005, 2010, 2015]
# interval_end_years = [2024]
# interval_end_years = [2016, 2017, 2018]
# interval_end_years = [2019, 2020, 2021, 2022, 2023]
# interval_end_years = [2005, 2010, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Creates dataframe of state_node codes and meanings
state_node_df = create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet)
node_codes = np.array(list(state_node_df['state_nodes']), dtype=np.uint32)

# Open all inputs not from the model here
print("Opening zarrs for non-model inputs")
pixel_area = xr.open_zarr(pixel_area_zarr_name).band_data
adm0 = xr.open_zarr(adm0_zarr_name).band_data
primary_forest_IFL = xr.open_zarr(primary_forest_IFL_zarr_name).band_data

# ADD ALL CONTEXTUAL LAYER NAMES HERE. THEY ARE USED IN DATAFRAME CREATION.
contextual_layer_names = ['state_nodes', 'gadm_adm0', 'primary_forest_IFL']

# Zonal stats iterates through model intervals (i.e. each interval analyzed separately)
for interval_end_year in interval_end_years:

    # Sets the interval name based on the interval type (5-year or annual)
    if interval_end_year in [2005, 2010, 2015]:
        interval = f"{interval_end_year-4}_{interval_end_year}"
    else:
        interval = f"{interval_end_year-1}_{interval_end_year}"

    print(f"Processing {interval}: {timestr()}")
    interval_start_time = time.time()

    # Creates a Pandas series of s3 uris for each model output layer.
    # This is only used to extract the pattern from each analysis layer, not to get data from them. 
    gross_emis_CO2_folder_interval = gross_emis_CO2_folder.replace("INTERVAL", interval)
    gross_emis_CO2_uris = list_folder_uris(gross_emis_CO2_folder_interval)
    gross_emis_all_gases_folder_interval = gross_emis_all_gases_folder.replace("INTERVAL", interval)
    gross_emis_all_gases_uris = list_folder_uris(gross_emis_all_gases_folder_interval)
    gross_remv_all_pools_folder_interval = gross_remv_all_pools_folder.replace("INTERVAL", interval)
    gross_remv_all_pools_uris = list_folder_uris(gross_remv_all_pools_folder_interval)
    net_flux_all_pools_CO2_folder_interval = net_flux_all_pools_CO2_folder.replace("INTERVAL", interval)
    net_flux_all_pools_CO2_uris = list_folder_uris(net_flux_all_pools_CO2_folder_interval)
    net_flux_all_pools_all_gases_folder_interval = net_flux_all_pools_all_gases_folder.replace("INTERVAL", interval)
    net_flux_all_pools_all_gases_uris = list_folder_uris(net_flux_all_pools_all_gases_folder_interval)
    node_folder_interval = node_folder.replace("INTERVAL", interval)
    node_tile_year_uris = list_folder_uris(node_folder_interval)

    # Gets the filename pattern for each model output layer
    gross_emis_CO2_output_pattern = parse_pattern_from_uri(gross_emis_CO2_uris)
    gross_emis_all_gases_output_pattern = parse_pattern_from_uri(gross_emis_all_gases_uris)
    gross_remv_all_pools_output_pattern = parse_pattern_from_uri(gross_remv_all_pools_uris)
    net_flux_CO2_output_pattern = parse_pattern_from_uri(net_flux_all_pools_CO2_uris)
    net_flux_all_gases_output_pattern = parse_pattern_from_uri(net_flux_all_pools_all_gases_uris)
    node_output_pattern = parse_pattern_from_uri(node_tile_year_uris)

    # # Version if not using zarrs for inputs. Reads the geotifs into chunks directly, rather than reading from pre-created zarrs. 
    # # This is slower than pre-creating zarrs and then reading from them, but leaving it here for reference. 
    # print(f"   Reading gross emis CO2 only for {interval}: {timestr()}")
    # gross_emis_CO2 = make_xarray_chunks(gross_emis_CO2_uris, chunk_size)['band_data']
    # print(f"   Reading gross emis all gases for {interval}: {timestr()}")
    # gross_emis_all_gases = make_xarray_chunks(gross_emis_all_gases_uris, chunk_size)['band_data']
    # print(f"   Reading gross removals for {interval}: {timestr()}")
    # gross_remv_all_pools = make_xarray_chunks(gross_remv_all_pools_uris, chunk_size)['band_data']   
    # print(f"   Reading net flux CO2 only for {interval}: {timestr()}")
    # net_flux_all_pools_CO2 = make_xarray_chunks(net_flux_all_pools_CO2_uris, chunk_size)['band_data']
    # print(f"   Reading net flux all_gases only for {interval}: {timestr()}")
    # net_flux_all_pools_all_gases = make_xarray_chunks(net_flux_all_pools_all_gases_uris, chunk_size)['band_data']   
    # print(f"   Reading state_nodes for {interval}: {timestr()}")
    # state_nodes = make_xarray_chunks(node_tile_year_uris, chunk_size)['band_data']  
    # state_nodes = state_nodes.astype('uint32')  # state_nodes should be uint32 but make_xarray_chunks makes it float64 for some reason

    # Locations of zarrs to be read
    gross_emis_CO2_zarr_name = f"{zarr_s3_path}{interval}/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_{interval}.zarr"
    gross_emis_all_gases_zarr_name = f"{zarr_s3_path}{interval}/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_{interval}.zarr"
    gross_remv_all_pools_zarr_name = f"{zarr_s3_path}{interval}/gross_removals__all_C_pools__MgCO2_ha_yr_{interval}.zarr"
    net_flux_all_pools_CO2_zarr_name = f"{zarr_s3_path}{interval}/net_flux__all_C_pools__CO2_only__MgCO2_ha_yr_{interval}.zarr"
    net_flux_all_pools_all_gases_zarr_name = f"{zarr_s3_path}{interval}/net_flux__all_C_pools__all_gases__MgCO2e_ha_yr_{interval}.zarr"
    node_zarr_name = f"{zarr_s3_path}{interval}/land_state_node_{interval}.zarr"

    # Reads zarrs
    print(f"   Reading {gross_emis_CO2_zarr_name} zar for gross emis CO2 only for {interval}: {timestr()}")
    gross_emis_CO2 = xr.open_zarr(gross_emis_CO2_zarr_name).band_data
    print(f"   Reading {gross_emis_all_gases_zarr_name} zar for gross emis all gases only for {interval}: {timestr()}")
    gross_emis_all_gases = xr.open_zarr(gross_emis_all_gases_zarr_name).band_data
    print(f"   Reading {gross_remv_all_pools_zarr_name} zar for gross removals for {interval}: {timestr()}")
    gross_remv_all_pools= xr.open_zarr(gross_remv_all_pools_zarr_name).band_data
    print(f"   Reading {net_flux_all_pools_CO2_zarr_name} zar for net flux CO2 only for {interval}: {timestr()}")
    net_flux_all_pools_CO2 = xr.open_zarr(net_flux_all_pools_CO2_zarr_name).band_data
    print(f"   Reading {net_flux_all_pools_all_gases_zarr_name} zar for net flux all_gases for {interval}: {timestr()}")
    net_flux_all_pools_all_gases = xr.open_zarr(net_flux_all_pools_all_gases_zarr_name).band_data
    print(f"   Reading {node_zarr_name} zar for nodes for {interval}: {timestr()}")
    state_nodes = xr.open_zarr(node_zarr_name).band_data


    # Makes all inputs align with the state_node extent (i.e. analysis limited to state_node extent, not extent of e.g., GADM).
    # Spent a while on this with ChatGPT: https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/684749fe-7b30-800a-ba8b-c502377f2c3a
    print(f"   Aligning {interval}: {timestr()}")
    reference = state_nodes  # All inputs get aligned with and cropeed to state_nodes
    state_nodes_aligned = reference  # TODO This can probably be consolidated with the line above.
        
    # Crops all inputs to state_nodes
    # Contextual layers
    adm0_aligned = safe_crop(adm0, reference)
    primary_forest_IFL_aligned = safe_crop(primary_forest_IFL, reference)

    # Analysis layers
    pixel_area_aligned = safe_crop(pixel_area, reference)
    gross_emis_CO2_aligned = safe_crop(gross_emis_CO2, reference)
    gross_emis_all_gases_aligned = safe_crop(gross_emis_all_gases, reference)
    gross_remv_all_pools_aligned = safe_crop(gross_remv_all_pools, reference)
    net_flux_all_pools_CO2_aligned = safe_crop(net_flux_all_pools_CO2, reference)
    net_flux_all_pools_all_gases_aligned = safe_crop(net_flux_all_pools_all_gases, reference)

    # Analysis layers output from model.
    # This has to be in the same order as entries in the flux_cube creation below.
    analysis_layer_dict = {0: gross_emis_CO2_output_pattern, 1: gross_emis_all_gases_output_pattern, 2: gross_remv_all_pools_output_pattern, 
                           3: net_flux_CO2_output_pattern, 4: net_flux_all_gases_output_pattern, 5: "pixel_area__ha"}

    # Makes a chunk of pixel areas (converts m^2 to ha)
    pixel_area__ha = (pixel_area_aligned / 10000).astype("float32").rename("pixel_area__ha")
    
    flux_cube = xr.DataArray(
        dask.array.stack([
            (gross_emis_CO2_aligned.data * pixel_area__ha.data).astype("float32"),
            (gross_emis_all_gases_aligned.data * pixel_area__ha.data).astype("float32"),
            (gross_remv_all_pools_aligned.data * pixel_area__ha.data).astype("float32"),
            (net_flux_all_pools_CO2_aligned.data * pixel_area__ha.data).astype("float32"),
            (net_flux_all_pools_all_gases_aligned.data * pixel_area__ha.data).astype("float32"),
            pixel_area__ha.data,
        ]),
        dims=("analysis_layer", "y", "x"),
    )
    # print(flux_cube)


    # Further alignment necessary in cases where zarr is made of non-contiguous geotifs.
    # Doesn't seem to hurt performance in general.
    flux_cube, adm0_aligned, state_nodes_aligned, primary_forest_IFL_aligned = xr.align(
        flux_cube, adm0_aligned, state_nodes_aligned, primary_forest_IFL_aligned,
        join="override"
    )


    # Each contextual layer has to have a unique name.  
    # Necessary to keep flox from getting confused about having multiple band_data to work with, per Solomon in Slack (2025-06-04). 
    # Any other contextual layers need to be renamed here, too. 
    adm0_aligned.name = "gadm_adm0"
    primary_forest_IFL_aligned.name = "primary_forest_IFL"
    state_nodes_aligned.name = "state_nodes"

    # Also names the analysis layers
    pixel_area__ha.name = "pixel_area__ha"
    gross_emis_CO2.name = "gross_emis_CO2"
    gross_emis_all_gases.name = "gross_emis_all_gases"
    gross_remv_all_pools.name = "gross_remv_all_pools"
    net_flux_all_pools_CO2.name = "net_flux_all_pools_CO2_only"
    net_flux_all_pools_all_gases.name = "net_flux_all_pools_all_gases"

    # All layers being used for zonal statistics, only used for printing metadata about them
    input_layers = [
        adm0_aligned,
        primary_forest_IFL_aligned,
        state_nodes_aligned,
        pixel_area__ha,
        gross_emis_CO2,
        gross_emis_all_gases,
        gross_remv_all_pools,
        net_flux_all_pools_CO2,
        net_flux_all_pools_all_gases
    ]

    # # Prints metadata of the xarray datasets being analyzed
    # for ds in input_layers:
    #     print(f"   Metadata for {ds.name} for {interval}")
    #     print(f"     X, Y resolution: {np.diff(ds.x.values).mean(), np.diff(ds.y.values).mean()}")
    #     print(f"     X range: {ds.x.values[[0, -1]]}")
    #     print(f"     Y range: {ds.y.values[[0, -1]]}")
    #     print(f"     Shape (x, y): {ds.sizes['x'], ds.sizes['y']}")
    #     print(f"     Data type: {ds.dtype}")

    
    print(f"   Computing {interval}: {timestr()}")
    flux_results = xarray_reduce(
        flux_cube,  # Layers to be analyzed
        # pixel_area_aligned,  # to test pixel_area as the only analysis layer
        *(adm0_aligned, state_nodes_aligned, primary_forest_IFL_aligned),  # Contextual layers
        func='sum',
        expected_groups=(gadm_adm0_ids, node_codes, primary_forest_IFL_codes),  # Contextual layer possible values. Must be in some order as contextual layers above. 
        reindex=ReindexStrategy(
            blockwise=False,
            array_type=ReindexArrayType.SPARSE_COO
        ),
        fill_value=0
    ).compute()

    
    # Prepares outputs for conversion into dataframe
    coord_dict = convert_to_coord_dict(flux_results)
    # print(coord_dict)
    
    # Creates the dataframe for the interval and does some processing of it
    df = create_interval_df(coord_dict, state_node_df)
    # print(df)

    # Calculates flux densities from the total fluxes
    df = calculate_interval_flux_densities(df)
    # print(df)

    # Combines dataframe from this interval with dataframes from previous intervals
    combined_df = pd.concat([combined_df, df])

    interval_end_time = time.time()
    print(f"   {interval} took {round(interval_end_time - interval_start_time)} seconds")


combined_df = combined_df.reset_index(drop=True)

analysis_end_time = time.time()
print(f"Analysis took {round(analysis_end_time - analysis_start_time)} seconds")

print(combined_df)

client.shutdown()

Opening zarrs for non-model inputs
Processing 2001_2005: 20250811_13_48_49
   Reading s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_4_2_1x1_inputs_per_ha/zarr/20250806/2001_2005/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2001_2005.zarr zar for gross emis CO2 only for 2001_2005: 20250811_13_48_49
   Reading s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_4_2_1x1_inputs_per_ha/zarr/20250806/2001_2005/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2001_2005.zarr zar for gross emis all gases only for 2001_2005: 20250811_13_48_49
   Reading s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_4_2_1x1_inputs_per_ha/zarr/20250806/2001_2005/gross_removals__all_C_pools__MgCO2_ha_yr_2001_2005.zarr zar for gross removals for 2001_2005: 20250811_13_48_49
   Reading s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_4_2_1x1_inputs_per_ha/zarr/20250806/2001_2005/net_flux__all_C_pools__CO2_only__MgCO2_ha_yr_2001_2005.zarr z

In [ ]:
# Gross emissions all pools all gases 2005 should = 458509003
# Area 2005 should = 206902007
# Net flux all pools all gases 2010 should = 94761495
print(combined_df[(combined_df.analysis_layer == 'gross_emissions__all_C_pools__all_gases__MgCO2e') & (combined_df.interval_end == 2005)]['value'].sum().round())
print(combined_df[(combined_df.analysis_layer == 'pixel_area__ha') & (combined_df.interval_end == 2005)]['value'].sum().round())
print(combined_df[(combined_df.analysis_layer == 'net_flux__all_C_pools__all_gases__MgCO2e') & (combined_df.interval_end == 2010)]['value'].sum().round())

In [ ]:
combined_df_wide = combined_df.pivot(index=['state_nodes', 'broad_class', 'detailed_class', 'interval_end', 'gadm_adm0', 'primary_forest_IFL', 'meaning'], columns="analysis_layer", values="value").reset_index()
# combined_df_wide

In [ ]:
# combined_df[(combined_df.analysis_layer == gross_remv_all_pools_output_pattern) 
# & (combined_df.gadm_adm0 == 180)]
# combined_df[(combined_df.state_node == 'n1110000') & (combined_df.interval_end == 2016) & (combined_df.primary_forest_IFL == 0) & (combined_df.gadm_adm0 == 178)]
combined_df[(combined_df.state_nodes == 1110000) & (combined_df.interval_end == 2016) & (combined_df.primary_forest_IFL == 0) & (combined_df.gadm_adm0 == 178)]

In [ ]:
df[(df.state_nodes == 1110000) & (df.primary_forest_IFL == 0) & (df.gadm_adm0 == 178)]

In [ ]:
# Export to a csv so data can be used in Excel or reused
combined_df_wide.to_csv('Cerrado_model_v0_4_2__2000_2024__20250807.csv', index=False)

In [ ]:
walker = pyg.walk(combined_df_wide)

In [ ]:
vis_spec = r"""{"config":[{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_5Dux","fid":"state_node","name":"state_node","basename":"state_node","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw__GB7","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"},{"dragId":"gw_Xb3-","fid":"node_grp","name":"node_grp","basename":"node_grp","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_IGtN","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_848c","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"dragId":"gw_y99d","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_Sjqt","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_5GlR","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_axS0","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e","name":"gross_emissions__all_C_pools__all_gases__MgCO2e","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_GZnF","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_cuXy","fid":"gross_removals__all_C_pools__MgCO2","name":"gross_removals__all_C_pools__MgCO2","basename":"gross_removals__all_C_pools__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_t1MS","fid":"gross_removals__all_C_pools__MgCO2__C_per_ha","name":"gross_removals__all_C_pools__MgCO2__C_per_ha","basename":"gross_removals__all_C_pools__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_12Yi","fid":"net_flux__all_C_pools__CO2_only__MgCO2","name":"net_flux__all_C_pools__CO2_only__MgCO2","basename":"net_flux__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_e6y_","fid":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_8-h6","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_uU2i","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"}],"color":[{"dragId":"gw_DITZ","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_GO_f","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension","rule":{"type":"one of","value":[180]}},{"dragId":"gw_c6Cy","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_wEzF","name":"Emissions"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_5Dux","fid":"state_node","name":"state_node","basename":"state_node","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw__GB7","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"},{"dragId":"gw_Xb3-","fid":"node_grp","name":"node_grp","basename":"node_grp","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_IGtN","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_848c","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"dragId":"gw_y99d","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_Sjqt","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_5GlR","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_axS0","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e","name":"gross_emissions__all_C_pools__all_gases__MgCO2e","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_GZnF","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_cuXy","fid":"gross_removals__all_C_pools__MgCO2","name":"gross_removals__all_C_pools__MgCO2","basename":"gross_removals__all_C_pools__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_t1MS","fid":"gross_removals__all_C_pools__MgCO2__C_per_ha","name":"gross_removals__all_C_pools__MgCO2__C_per_ha","basename":"gross_removals__all_C_pools__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_12Yi","fid":"net_flux__all_C_pools__CO2_only__MgCO2","name":"net_flux__all_C_pools__CO2_only__MgCO2","basename":"net_flux__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_e6y_","fid":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_fa0j","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_uU2i","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"}],"color":[{"dragId":"gw_DITZ","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_GO_f","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension","rule":{"type":"one of","value":[180]}},{"dragId":"gw_c6Cy","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_CA7R","name":"Emission factor"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_5Dux","fid":"state_node","name":"state_node","basename":"state_node","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw__GB7","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"},{"dragId":"gw_Xb3-","fid":"node_grp","name":"node_grp","basename":"node_grp","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_IGtN","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_848c","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"dragId":"gw_y99d","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_Sjqt","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_5GlR","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_axS0","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e","name":"gross_emissions__all_C_pools__all_gases__MgCO2e","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_GZnF","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_cuXy","fid":"gross_removals__all_C_pools__MgCO2","name":"gross_removals__all_C_pools__MgCO2","basename":"gross_removals__all_C_pools__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_t1MS","fid":"gross_removals__all_C_pools__MgCO2__C_per_ha","name":"gross_removals__all_C_pools__MgCO2__C_per_ha","basename":"gross_removals__all_C_pools__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_12Yi","fid":"net_flux__all_C_pools__CO2_only__MgCO2","name":"net_flux__all_C_pools__CO2_only__MgCO2","basename":"net_flux__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_e6y_","fid":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_fCJt","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_uU2i","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"}],"color":[{"dragId":"gw_DITZ","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_GO_f","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension","rule":{"type":"one of","value":[180]}},{"dragId":"gw_c6Cy","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_iAyO","name":"Emissions area"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_5Dux","fid":"state_node","name":"state_node","basename":"state_node","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw__GB7","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"},{"dragId":"gw_Xb3-","fid":"node_grp","name":"node_grp","basename":"node_grp","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_IGtN","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_848c","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"dragId":"gw_y99d","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_Sjqt","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_5GlR","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_axS0","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e","name":"gross_emissions__all_C_pools__all_gases__MgCO2e","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_GZnF","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_cuXy","fid":"gross_removals__all_C_pools__MgCO2","name":"gross_removals__all_C_pools__MgCO2","basename":"gross_removals__all_C_pools__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_t1MS","fid":"gross_removals__all_C_pools__MgCO2__C_per_ha","name":"gross_removals__all_C_pools__MgCO2__C_per_ha","basename":"gross_removals__all_C_pools__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_12Yi","fid":"net_flux__all_C_pools__CO2_only__MgCO2","name":"net_flux__all_C_pools__CO2_only__MgCO2","basename":"net_flux__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_e6y_","fid":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_rnBz","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_7xrz","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_fCJt","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_uU2i","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"}],"color":[{"dragId":"gw_DITZ","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_GO_f","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension","rule":{"type":"one of","value":[180]}},{"dragId":"gw_c6Cy","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_MCCS","name":"All three panels"}],"chart_map":{},"workflow_list":[{"workflow":[{"type":"filter","filters":[{"fid":"gadm_adm0","rule":{"type":"one of","value":[180]}},{"fid":"meaning","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["interval_end","meaning"],"measures":[{"field":"gross_emissions__all_C_pools__CO2_only__MgCO2","agg":"sum","asFieldKey":"gross_emissions__all_C_pools__CO2_only__MgCO2_sum"}]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"gadm_adm0","rule":{"type":"one of","value":[180]}},{"fid":"meaning","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["interval_end","meaning"],"measures":[{"field":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","agg":"sum","asFieldKey":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha_sum"}]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"gadm_adm0","rule":{"type":"one of","value":[180]}},{"fid":"meaning","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["interval_end","meaning"],"measures":[{"field":"area__ha","agg":"sum","asFieldKey":"area__ha_sum"}]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"gadm_adm0","rule":{"type":"one of","value":[180]}},{"fid":"meaning","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["interval_end","meaning"],"measures":[{"field":"gross_emissions__all_C_pools__CO2_only__MgCO2","agg":"sum","asFieldKey":"gross_emissions__all_C_pools__CO2_only__MgCO2_sum"},{"field":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","agg":"sum","asFieldKey":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha_sum"},{"field":"area__ha","agg":"sum","asFieldKey":"area__ha_sum"}]}]}]}],"timezoneOffsetSeconds":-14400,"version":"0.3.17"}"""
pyg.walk(combined_df_wide, spec=vis_spec)